# imports

In [1]:
import itertools
import json

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)

RANDOM_STATE = 42

# EDA

In [2]:
raw_features = pd.read_csv("train_features.csv")
raw_features.head()

,match_id_hash,game_time,game_mode,lobby_type,objectives_len,chat_len,r1_hero_id,r1_kills,r1_deaths,r1_assists,r1_denies,r1_gold,r1_lh,r1_xp,r1_health,r1_max_health,r1_max_mana,r1_level,r1_x,r1_y,r1_stuns,r1_creeps_stacked,r1_camps_stacked,r1_rune_pickups,r1_firstblood_claimed,r1_teamfight_participation,r1_towers_killed,r1_roshans_killed,r1_obs_placed,r1_sen_placed,r2_hero_id,r2_kills,r2_deaths,r2_assists,r2_denies,r2_gold,r2_lh,r2_xp,r2_health,r2_max_health,r2_max_mana,r2_level,r2_x,r2_y,r2_stuns,r2_creeps_stacked,r2_camps_stacked,r2_rune_pickups,r2_firstblood_claimed,r2_teamfight_participation,r2_towers_killed,r2_roshans_killed,r2_obs_placed,r2_sen_placed,r3_hero_id,r3_kills,r3_deaths,r3_assists,r3_denies,r3_gold,r3_lh,r3_xp,r3_health,r3_max_health,r3_max_mana,r3_level,r3_x,r3_y,r3_stuns,r3_creeps_stacked,r3_camps_stacked,r3_rune_pickups,r3_firstblood_claimed,r3_teamfight_participation,r3_towers_killed,r3_roshans_killed,r3_obs_placed,r3_sen_placed,r4_hero_id,r4_kills,r4_deaths,r4_assists,r4_denies,r4_gold,r4_lh,r4_xp,r4_health,r4_max_health,r4_max_mana,r4_level,r4_x,r4_y,r4_stuns,r4_creeps_stacked,r4_camps_stacked,r4_rune_pickups,r4_firstblood_claimed,r4_teamfight_participation,r4_towers_killed,r4_roshans_killed,r4_obs_placed,r4_sen_placed,r5_hero_id,r5_kills,r5_deaths,r5_assists,r5_denies,r5_gold,r5_lh,r5_xp,r5_health,r5_max_health,r5_max_mana,r5_level,r5_x,r5_y,r5_stuns,r5_creeps_stacked,r5_camps_stacked,r5_rune_pickups,r5_firstblood_claimed,r5_teamfight_participation,r5_towers_killed,r5_roshans_killed,r5_obs_placed,r5_sen_placed,d1_hero_id,d1_kills,d1_deaths,d1_assists,d1_denies,d1_gold,d1_lh,d1_xp,d1_health,d1_max_health,d1_max_mana,d1_level,d1_x,d1_y,d1_stuns,d1_creeps_stacked,d1_camps_stacked,d1_rune_pickups,d1_firstblood_claimed,d1_teamfight_participation,d1_towers_killed,d1_roshans_killed,d1_obs_placed,d1_sen_placed,d2_hero_id,d2_kills,d2_deaths,d2_assists,d2_denies,d2_gold,d2_lh,d2_xp,d2_health,d2_max_health,d2_max_mana,d2_level,d2_x,d2_y,d2_stuns,d2_creeps_stacked,d2_camps_stacked,d2_rune_pickups,d2_firstblood_claimed,d2_teamfight_participation,d2_towers_killed,d2_roshans_killed,d2_obs_placed,d2_sen_placed,d3_hero_id,d3_kills,d3_deaths,d3_assists,d3_denies,d3_gold,d3_lh,d3_xp,d3_health,d3_max_health,d3_max_mana,d3_level,d3_x,d3_y,d3_stuns,d3_creeps_stacked,d3_camps_stacked,d3_rune_pickups,d3_firstblood_claimed,d3_teamfight_participation,d3_towers_killed,d3_roshans_killed,d3_obs_placed,d3_sen_placed,d4_hero_id,d4_kills,d4_deaths,d4_assists,d4_denies,d4_gold,d4_lh,d4_xp,d4_health,d4_max_health,d4_max_mana,d4_level,d4_x,d4_y,d4_stuns,d4_creeps_stacked,d4_camps_stacked,d4_rune_pickups,d4_firstblood_claimed,d4_teamfight_participation,d4_towers_killed,d4_roshans_killed,d4_obs_placed,d4_sen_placed,d5_hero_id,d5_kills,d5_deaths,d5_assists,d5_denies,d5_gold,d5_lh,d5_xp,d5_health,d5_max_health,d5_max_mana,d5_level,d5_x,d5_y,d5_stuns,d5_creeps_stacked,d5_camps_stacked,d5_rune_pickups,d5_firstblood_claimed,d5_teamfight_participation,d5_towers_killed,d5_roshans_killed,d5_obs_placed,d5_sen_placed
0,a400b8f29dece5f4d266f49f1ae2e98a,155,22,7,1,11,11,0,0,0,0,543,7,533,358,600,350.93784,2,116,122,0.000000,0,0,1,0,0.000000,0,0,0,0,78,0,0,0,3,399,4,478,636,720,254.93774,2,124,126,0.000000,0,0,0,0,0.000000,0,0,0,0,14,0,1,0,0,304,0,130,700,700,242.93773,1,70,156,0.000000,0,0,1,0,0.000000,0,0,0,0,59,0,0,0,1,389,4,506,399,700,326.93780,2,170,86,0.000000,0,0,0,0,0.000000,0,0,0,0,77,0,0,0,0,402,10,344,422,800,314.93780,2,120,100,0.000000,0,0,0,0,0.000000,0,0,0,0,12,0,0,1,13,982,12,780,650,720,386.93787,3,82,170,0.000000,0,0,1,0,1.00,0,0,0,0,21,0,0,0,6,788,9,706,640,640,422.93790,3,174,90,0.000000,0,0,2,0,0.00,0,0,0,0,60,0,0,0,1,531,0,307,720,720,242.93773,2,180,84,0.299948,0,0,2,0,0.00,0,0,0,0,84,1,0,0,0,796,0,421,760,760,326.93780,2,90,150,0.000000,0,0,2,1,1.0,0,0,1,0,34,0,0,0,0,851,11,870,593,680,566.93805,3,128,128,0.000000,0,0,0,0,0.00,0,0,0,0
1,b9c57c450ce74a2af79c9ce96fac144d,658,4,0,3,10,15,7,2,0,7,5257,52,3937,1160,1160,566.93805,8,

In [3]:
raw_features['lobby_type'].value_counts()

lobby_type
7    27049
0    12626
Name: count, dtype: int64

In [4]:
raw_targets = pd.read_csv("train_targets.csv")
raw_targets.head()

,match_id_hash,game_time,radiant_win,duration,time_remaining,next_roshan_team
0,a400b8f29dece5f4d266f49f1ae2e98a,155,False,992,837,NaN
1,b9c57c450ce74a2af79c9ce96fac144d,658,True,1154,496,NaN
2,6db558535151ea18ca70a6892197db41,21,True,1503,1482,Radiant
3,46a0ddce8f7ed2a8d9bd5edcbb925682,576,True,1952,1376,NaN
4,b1b35ff97723d9b7ade1c9c3cf48f770,453,False,2001,1548,NaN


In [21]:
def create_features(df):

    # radiant_coords = [f'r{i}_{j}' for i in range(1, 6) for j in ('x','y')]
    # dire_coords = [f'd{i}_{j}' for i in range(1, 6) for j in ('x','y')]
    
    # df['r_centr_x'] = df[radiant_coords].apply(lambda x: geometry(x.tolist(), flag=2).centroid.x, axis=1)
    # df['r_centr_y'] = df[radiant_coords].apply(lambda x: geometry(x.tolist(), flag=2).centroid.y, axis=1)
    # df['d_centr_x'] = df[dire_coords].apply(lambda x: geometry(x.tolist(), flag=2).centroid.x, axis=1)
    # df['d_centr_y'] = df[dire_coords].apply(lambda x: geometry(x.tolist(), flag=2).centroid.y, axis=1)
    # df['r_repr_x'] = df[radiant_coords].apply(lambda x: geometry(x.tolist(), flag=2).buffer(1).representative_point().x, axis=1)
    # df['r_repr_y'] = df[radiant_coords].apply(lambda x: geometry(x.tolist(), flag=2).buffer(1).representative_point().y, axis=1)
    # df['d_repr_x'] = df[dire_coords].apply(lambda x: geometry(x.tolist(), flag=2).buffer(1).representative_point().x, axis=1)
    # df['d_repr_y'] = df[dire_coords].apply(lambda x: geometry(x.tolist(), flag=2).buffer(1).representative_point().y, axis=1)
    # df['r_area'] = df[radiant_coords].apply(lambda x: geometry(x.tolist(), flag=1), axis=1)
    # df['r_perim'] = df[radiant_coords].apply(lambda x: geometry(x.tolist(), flag=0), axis=1)
    # df['d_area'] = df[dire_coords].apply(lambda x: geometry(x.tolist(), flag=1), axis=1)
    # df['d_perim'] = df[dire_coords].apply(lambda x: geometry(x.tolist(), flag=0), axis=1)
    # df['intersection'] = df[radiant_coords + dire_coords].apply(lambda x: intersection(x.iloc[:10].tolist(), x.iloc[10:].tolist()), axis=1)
    # df['hausdorff_distance'] = df[radiant_coords + dire_coords].apply(lambda x: hausdorff_distance(x.iloc[:10].tolist(), x.iloc[10:].tolist()), axis=1)
    # df['area_diff'] = df['r_area'] - df['d_area']
    # df['perim_diff'] = df['r_perim'] - df['d_perim']
    
    # Суммарные характеристики команды "Свет"
    df["radiant_total_gold"] = df[[f"r{i}_gold" for i in range(1, 6)]].sum(axis=1)
    df["radiant_total_kills"] = df[[f"r{i}_kills" for i in range(1, 6)]].sum(axis=1)
    df["radiant_total_deaths"] = df[[f"r{i}_deaths" for i in range(1, 6)]].sum(axis=1)
    df["radiant_total_level"] = df[[f"r{i}_level" for i in range(1, 6)]].sum(axis=1)

    # Суммарные характеристики команды "Тьма"
    df["dire_total_gold"] = df[[f"d{i}_gold" for i in range(1, 6)]].sum(axis=1)
    df["dire_total_kills"] = df[[f"d{i}_kills" for i in range(1, 6)]].sum(axis=1)
    df["dire_total_deaths"] = df[[f"d{i}_deaths" for i in range(1, 6)]].sum(axis=1)
    df["dire_total_level"] = df[[f"d{i}_level" for i in range(1, 6)]].sum(axis=1)

    # Разницы между командами
    df["radiant_gold_advantage"] = df["radiant_total_gold"] - df["dire_total_gold"]
    df["radiant_kill_advantage"] = df["radiant_total_kills"] - df["dire_total_kills"]
    df["radiant_level_advantage"] = df["radiant_total_level"] - df["dire_total_level"]

    # Общее количество убийств и смертей
    df["total_kills_deaths"] = (
        df["radiant_total_kills"] + df["dire_total_kills"] +
        df["radiant_total_deaths"] + df["dire_total_deaths"]
    )

    
    return df



In [22]:
df_feats = create_features(raw_features)

In [26]:
df_feats

,match_id_hash,game_time,game_mode,lobby_type,objectives_len,chat_len,r1_hero_id,r1_kills,r1_deaths,r1_assists,r1_denies,r1_gold,r1_lh,r1_xp,r1_health,r1_max_health,r1_max_mana,r1_level,r1_x,r1_y,r1_stuns,r1_creeps_stacked,r1_camps_stacked,r1_rune_pickups,r1_firstblood_claimed,r1_teamfight_participation,r1_towers_killed,r1_roshans_killed,r1_obs_placed,r1_sen_placed,r2_hero_id,r2_kills,r2_deaths,r2_assists,r2_denies,r2_gold,r2_lh,r2_xp,r2_health,r2_max_health,r2_max_mana,r2_level,r2_x,r2_y,r2_stuns,r2_creeps_stacked,r2_camps_stacked,r2_rune_pickups,r2_firstblood_claimed,r2_teamfight_participation,r2_towers_killed,r2_roshans_killed,r2_obs_placed,r2_sen_placed,r3_hero_id,r3_kills,r3_deaths,r3_assists,r3_denies,r3_gold,r3_lh,r3_xp,r3_health,r3_max_health,r3_max_mana,r3_level,r3_x,r3_y,r3_stuns,r3_creeps_stacked,r3_camps_stacked,r3_rune_pickups,r3_firstblood_claimed,r3_teamfight_participation,r3_towers_killed,r3_roshans_killed,r3_obs_placed,r3_sen_placed,r4_hero_id,r4_kills,r4_deaths,r4_assists,r4_denies,r4_gold,r4_lh,r4_xp,r4_health,r4_max_health,r4_max_mana,r4_level,r4_x,r4_y,r4_stuns,r4_creeps_stacked,r4_camps_stacked,r4_rune_pickups,r4_firstblood_claimed,r4_teamfight_participation,r4_towers_killed,r4_roshans_killed,r4_obs_placed,r4_sen_placed,r5_hero_id,r5_kills,r5_deaths,r5_assists,r5_denies,r5_gold,r5_lh,r5_xp,r5_health,r5_max_health,r5_max_mana,r5_level,r5_x,r5_y,r5_stuns,r5_creeps_stacked,r5_camps_stacked,r5_rune_pickups,r5_firstblood_claimed,r5_teamfight_participation,r5_towers_killed,r5_roshans_killed,r5_obs_placed,r5_sen_placed,d1_hero_id,d1_kills,d1_deaths,d1_assists,d1_denies,d1_gold,d1_lh,d1_xp,d1_health,d1_max_health,d1_max_mana,d1_level,d1_x,d1_y,d1_stuns,d1_creeps_stacked,d1_camps_stacked,d1_rune_pickups,d1_firstblood_claimed,d1_teamfight_participation,d1_towers_killed,d1_roshans_killed,d1_obs_placed,d1_sen_placed,d2_hero_id,d2_kills,d2_deaths,d2_assists,d2_denies,d2_gold,d2_lh,d2_xp,d2_health,d2_max_health,d2_max_mana,d2_level,d2_x,d2_y,d2_stuns,d2_creeps_stacked,d2_camps_stacked,d2_rune_pickups,d2_firstblood_claimed,d2_teamfight_participation,d2_towers_killed,d2_roshans_killed,d2_obs_placed,d2_sen_placed,d3_hero_id,d3_kills,d3_deaths,d3_assists,d3_denies,d3_gold,d3_lh,d3_xp,d3_health,d3_max_health,d3_max_mana,d3_level,d3_x,d3_y,d3_stuns,d3_creeps_stacked,d3_camps_stacked,d3_rune_pickups,d3_firstblood_claimed,d3_teamfight_participation,d3_towers_killed,d3_roshans_killed,d3_obs_placed,d3_sen_placed,d4_hero_id,d4_kills,d4_deaths,d4_assists,d4_denies,d4_gold,d4_lh,d4_xp,d4_health,d4_max_health,d4_max_mana,d4_level,d4_x,d4_y,d4_stuns,d4_creeps_stacked,d4_camps_stacked,d4_rune_pickups,d4_firstblood_claimed,d4_teamfight_participation,d4_towers_killed,d4_roshans_killed,d4_obs_placed,d4_sen_placed,d5_hero_id,d5_kills,d5_deaths,d5_assists,d5_denies,d5_gold,d5_lh,d5_xp,d5_health,d5_max_health,d5_max_mana,d5_level,d5_x,d5_y,d5_stuns,d5_creeps_stacked,d5_camps_stacked,d5_rune_pickups,d5_firstblood_claimed,d5_teamfight_participation,d5_towers_killed,d5_roshans_killed,d5_obs_placed,d5_sen_placed,r_NetWorth,d_NetWorth,r_Total_XP,d_Total_XP,r_score,d_score,r_stuns,d_stuns,radiant_total_gold,radiant_total_kills,radiant_total_deaths,radiant_total_level,dire_total_gold,dire_total_kills,dire_total_deaths,dire_total_level,radiant_gold_advantage,radiant_kill_advantage,radiant_level_advantage,total_kills_deaths
0,a400b8f29dece5f4d266f49f1ae2e98a,155,22,7,1,11,11,0,0,0,0,543,7,533,358,600,350.93784,2,116,122,0.000000,0,0,1,0,0.000000,0,0,0,0,78,0,0,0,3,399,4,478,636,720,254.93774,2,124,126,0.000000,0,0,0,0,0.000000,0,0,0,0,14,0,1,0,0,304,0,130,700,700,242.93773,1,70,156,0.000000,0,0,1,0,0.000000,0,0,0,0,59,0,0,0,1,389,4,506,399,700,326.93780,2,170,86,0.000000,0,0,0,0,0.000000,0,0,0,0,77,0,0,0,0,402,10,344,422,800,314.93780,2,120,100,0.000000,0,0,0,0,0.000000,0,0,0,0,12,0,0,1,13,982,12,780,650,720,386.93787,3,82,170,0.000000,0,0,1,0,1.00000,0,0,0,0,21,0,0,0,6,788,9,706,640,640,422.93790,3,174,90,0.000000,0,0,2,0,0.00000,0,0,0,0,60,0,0,0,1

In [ ]:
# raw_features['r_NetWorth'] = raw_features['r1_gold'] + raw_features['r2_gold'] + raw_features['r3_gold'] + raw_features['r4_gold'] + raw_features['r5_gold']
# raw_features['d_NetWorth'] = raw_features['d1_gold'] + raw_features['d2_gold'] + raw_features['d3_gold'] + raw_features['d4_gold'] + raw_features['d5_gold']

# raw_features['r_Total_XP'] = raw_features['r1_xp'] + raw_features['r2_xp'] + raw_features['r3_xp'] + raw_features['r4_xp'] + raw_features['r5_xp']
# raw_features['d_Total_XP'] = raw_features['d1_xp'] + raw_features['d2_xp'] + raw_features['d3_xp'] + raw_features['d4_xp'] + raw_features['d5_xp']


# raw_features['r_score'] = raw_features['r1_kills'] + raw_features['r2_kills'] + raw_features['r3_kills'] + raw_features['r4_kills'] + raw_features['r5_kills']
# raw_features['d_score'] = raw_features['d1_kills'] + raw_features['d2_kills'] + raw_features['d3_kills'] + raw_features['d4_kills'] + raw_features['d5_kills']

# raw_features['r_stuns'] = raw_features['r1_stuns'] + raw_features['r2_stuns'] + raw_features['r3_stuns'] + raw_features['r4_stuns'] + raw_features['r5_stuns']
# raw_features['d_stuns'] = raw_features['d1_stuns'] + raw_features['d2_stuns'] + raw_features['d3_stuns'] + raw_features['d4_stuns'] + raw_features['d5_stuns']

In [28]:
# X = raw_features.drop(["match_id_hash", "game_time", "lobby_type", "game_mode", "objectives_len", "chat_len"], axis=1)
X = df_feats.drop('match_id_hash', axis=1)
y = raw_targets["radiant_win"]

In [29]:
X

,game_time,game_mode,lobby_type,objectives_len,chat_len,r1_hero_id,r1_kills,r1_deaths,r1_assists,r1_denies,r1_gold,r1_lh,r1_xp,r1_health,r1_max_health,r1_max_mana,r1_level,r1_x,r1_y,r1_stuns,r1_creeps_stacked,r1_camps_stacked,r1_rune_pickups,r1_firstblood_claimed,r1_teamfight_participation,r1_towers_killed,r1_roshans_killed,r1_obs_placed,r1_sen_placed,r2_hero_id,r2_kills,r2_deaths,r2_assists,r2_denies,r2_gold,r2_lh,r2_xp,r2_health,r2_max_health,r2_max_mana,r2_level,r2_x,r2_y,r2_stuns,r2_creeps_stacked,r2_camps_stacked,r2_rune_pickups,r2_firstblood_claimed,r2_teamfight_participation,r2_towers_killed,r2_roshans_killed,r2_obs_placed,r2_sen_placed,r3_hero_id,r3_kills,r3_deaths,r3_assists,r3_denies,r3_gold,r3_lh,r3_xp,r3_health,r3_max_health,r3_max_mana,r3_level,r3_x,r3_y,r3_stuns,r3_creeps_stacked,r3_camps_stacked,r3_rune_pickups,r3_firstblood_claimed,r3_teamfight_participation,r3_towers_killed,r3_roshans_killed,r3_obs_placed,r3_sen_placed,r4_hero_id,r4_kills,r4_deaths,r4_assists,r4_denies,r4_gold,r4_lh,r4_xp,r4_health,r4_max_health,r4_max_mana,r4_level,r4_x,r4_y,r4_stuns,r4_creeps_stacked,r4_camps_stacked,r4_rune_pickups,r4_firstblood_claimed,r4_teamfight_participation,r4_towers_killed,r4_roshans_killed,r4_obs_placed,r4_sen_placed,r5_hero_id,r5_kills,r5_deaths,r5_assists,r5_denies,r5_gold,r5_lh,r5_xp,r5_health,r5_max_health,r5_max_mana,r5_level,r5_x,r5_y,r5_stuns,r5_creeps_stacked,r5_camps_stacked,r5_rune_pickups,r5_firstblood_claimed,r5_teamfight_participation,r5_towers_killed,r5_roshans_killed,r5_obs_placed,r5_sen_placed,d1_hero_id,d1_kills,d1_deaths,d1_assists,d1_denies,d1_gold,d1_lh,d1_xp,d1_health,d1_max_health,d1_max_mana,d1_level,d1_x,d1_y,d1_stuns,d1_creeps_stacked,d1_camps_stacked,d1_rune_pickups,d1_firstblood_claimed,d1_teamfight_participation,d1_towers_killed,d1_roshans_killed,d1_obs_placed,d1_sen_placed,d2_hero_id,d2_kills,d2_deaths,d2_assists,d2_denies,d2_gold,d2_lh,d2_xp,d2_health,d2_max_health,d2_max_mana,d2_level,d2_x,d2_y,d2_stuns,d2_creeps_stacked,d2_camps_stacked,d2_rune_pickups,d2_firstblood_claimed,d2_teamfight_participation,d2_towers_killed,d2_roshans_killed,d2_obs_placed,d2_sen_placed,d3_hero_id,d3_kills,d3_deaths,d3_assists,d3_denies,d3_gold,d3_lh,d3_xp,d3_health,d3_max_health,d3_max_mana,d3_level,d3_x,d3_y,d3_stuns,d3_creeps_stacked,d3_camps_stacked,d3_rune_pickups,d3_firstblood_claimed,d3_teamfight_participation,d3_towers_killed,d3_roshans_killed,d3_obs_placed,d3_sen_placed,d4_hero_id,d4_kills,d4_deaths,d4_assists,d4_denies,d4_gold,d4_lh,d4_xp,d4_health,d4_max_health,d4_max_mana,d4_level,d4_x,d4_y,d4_stuns,d4_creeps_stacked,d4_camps_stacked,d4_rune_pickups,d4_firstblood_claimed,d4_teamfight_participation,d4_towers_killed,d4_roshans_killed,d4_obs_placed,d4_sen_placed,d5_hero_id,d5_kills,d5_deaths,d5_assists,d5_denies,d5_gold,d5_lh,d5_xp,d5_health,d5_max_health,d5_max_mana,d5_level,d5_x,d5_y,d5_stuns,d5_creeps_stacked,d5_camps_stacked,d5_rune_pickups,d5_firstblood_claimed,d5_teamfight_participation,d5_towers_killed,d5_roshans_killed,d5_obs_placed,d5_sen_placed,r_NetWorth,d_NetWorth,r_Total_XP,d_Total_XP,r_score,d_score,r_stuns,d_stuns,radiant_total_gold,radiant_total_kills,radiant_total_deaths,radiant_total_level,dire_total_gold,dire_total_kills,dire_total_deaths,dire_total_level,radiant_gold_advantage,radiant_kill_advantage,radiant_level_advantage,total_kills_deaths
0,155,22,7,1,11,11,0,0,0,0,543,7,533,358,600,350.93784,2,116,122,0.000000,0,0,1,0,0.000000,0,0,0,0,78,0,0,0,3,399,4,478,636,720,254.93774,2,124,126,0.000000,0,0,0,0,0.000000,0,0,0,0,14,0,1,0,0,304,0,130,700,700,242.93773,1,70,156,0.000000,0,0,1,0,0.000000,0,0,0,0,59,0,0,0,1,389,4,506,399,700,326.93780,2,170,86,0.000000,0,0,0,0,0.000000,0,0,0,0,77,0,0,0,0,402,10,344,422,800,314.93780,2,120,100,0.000000,0,0,0,0,0.000000,0,0,0,0,12,0,0,1,13,982,12,780,650,720,386.93787,3,82,170,0.000000,0,0,1,0,1.00000,0,0,0,0,21,0,0,0,6,788,9,706,640,640,422.93790,3,174,90,0.000000,0,0,2,0,0.00000,0,0,0,0,60,0,0,0,1,531,0,307,720,720,242.93773,2,180,84,0.299948,

In [10]:
y.head()

0    False
1     True
2     True
3     True
4    False
Name: radiant_win, dtype: bool

In [ ]:
# path = "DOTA 2//train_matches.jsonl"

# with open(path, "r", encoding="utf-8") as f:
#     for line in itertools.islice(f, 5):  # первые 5 строк
#         print(line.strip())

{"game_time": 155, "match_id_hash": "a400b8f29dece5f4d266f49f1ae2e98a", "teamfights": [], "objectives": [{"time": 124.203, "type": "CHAT_MESSAGE_FIRSTBLOOD", "slot": 8, "key": 2, "player_slot": 131}], "chat": [{"player_slot": 0, "time": -13.63, "text": "???"}, {"player_slot": 0, "time": -10.964, "text": "? ???? ??????"}, {"player_slot": 0, "time": -7.965, "text": "???????? ????("}, {"player_slot": 4, "time": 43.756, "text": "???"}, {"player_slot": 2, "time": 44.422, "text": "??? ??????"}, {"player_slot": 4, "time": 45.289, "text": "*[*[*["}, {"player_slot": 2, "time": 46.622, "text": "????"}, {"player_slot": 4, "time": 47.355, "text": "??? ??????"}, {"player_slot": 4, "time": 48.688, "text": "????"}, {"player_slot": 2, "time": 52.021, "text": "???????? ??? ??????"}, {"player_slot": 4, "time": 73.349, "text": "????? ??? ?????"}], "game_mode": 22, "lobby_type": 7, "players": [{"player_slot": 0, "hero_id": 11, "hero_name": "npc_dota_hero_nevermore", "account_id_hash": "86bf86c548874247868

In [ ]:
# src = "DOTA 2//train_matches.jsonl"
# dst = "train_matches_sample.jsonl"

# step = 1  # каждую сотую строку берём в семпл

# with open(src, "r", encoding="utf-8") as fin, open(dst, "w", encoding="utf-8") as fout:
#     for i, line in enumerate(fin, start=1):
#         if not line.strip():
#             continue
#         if i % step == 0:
#             fout.write(line)

# print("Готово, семпл сохранён в", dst)

Готово, семпл сохранён в train_matches_sample.jsonl


In [30]:
# df = pd.read_json("train_matches_sample.jsonl", lines=True)
# df.head()

In [31]:
df.shape

(33723, 9)

In [32]:
# df['players'][1]

# Catboost

In [33]:
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=7,
    loss_function='Logloss',
    eval_metric='AUC',
    random_state=RANDOM_STATE,
    verbose=100)


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,      
    random_state=RANDOM_STATE,    
    stratify=y,
    shuffle=True         
)

model.fit(X_train, y_train)#, eval_set=(X_test, y_test), verbose=100)

0:	total: 20.2ms	remaining: 20.2s
100:	total: 1.38s	remaining: 12.3s
200:	total: 2.64s	remaining: 10.5s
300:	total: 3.89s	remaining: 9.04s
400:	total: 5.13s	remaining: 7.67s
500:	total: 6.36s	remaining: 6.33s
600:	total: 7.58s	remaining: 5.03s
700:	total: 8.81s	remaining: 3.76s
800:	total: 10.1s	remaining: 2.51s
900:	total: 11.3s	remaining: 1.25s
999:	total: 12.5s	remaining: 0us


In [34]:
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy: {accuracy:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")

Accuracy: 0.7283
ROC AUC: 0.8174


In [18]:
y_pred

array([ True,  True, False, ..., False,  True,  True])

In [19]:
# model = CatBoostClassifier(
#     iterations=1000,
#     learning_rate=0.1,
#     depth=7,
#     loss_function='Logloss',
#     eval_metric='AUC',
#     random_state=RANDOM_STATE,
#     verbose=100)


# X_train, X_test, y_train, y_test = train_test_split(
#     X, y,
#     test_size=0.25,      
#     random_state=RANDOM_STATE,    
#     stratify=y,
#     shuffle=True         
# )

# model.fit(X_train, y_train)#, eval_set=(X_test, y_test), verbose=100)

# y_pred = model.predict(X_test)
# accuracy = accuracy_score(y_test, y_pred)
# roc_auc = roc_auc_score(y_test, y_pred)

# print(f"Accuracy: {accuracy:.4f}")
# print(f"ROC AUC: {roc_auc:.4f}")
